# Sistem Prediksi Diabetes Tipe 2 dengan Federated Learning dan Blockchain

## Abstrak

Diabetes mellitus tipe 2 merupakan salah satu penyakit kronis yang paling umum di seluruh dunia. Prediksi dini dan akurat sangat penting untuk mencegah komplikasi serius. Dalam penelitian ini, kami mengembangkan sistem prediksi diabetes yang menggabungkan **Federated Learning (FL)** dan **Blockchain** untuk memastikan:

1. **Privasi Pasien**: Data medis tidak pernah dikirim ke server pusat
2. **Keamanan**: Blockchain memastikan integritas model update
3. **Desentralisasi**: Training terjadi di perangkat masing-masing pasien
4. **Deteksi Serangan**: Sistem dapat mendeteksi poisoning attack

---

## Daftar Isi

1. [Pendahuluan](#pendahuluan)
2. [Metodologi](#metodologi)
3. [Implementasi](#implementasi)
4. [Hasil dan Pembahasan](#hasil)
5. [Kesimpulan](#kesimpulan)

<a name="pendahuluan"></a>
## 1. Pendahuluan

### 1.1 Latar Belakang

Diabetes mellitus adalah penyakit metabolik kronis yang ditandai dengan kadar glukosa darah tinggi. Menurut International Diabetes Federation, pada tahun 2021 terdapat sekitar 537 juta orang dewasa yang hidup dengan diabetes, dan angka ini diprediksi akan meningkat menjadi 783 juta pada tahun 2045.

### 1.2 Masalah

Sistem machine learning konvensional untuk prediksi diabetes memiliki beberapa kelemahan:

- **Masalah Privasi**: Data pasien harus dikumpulkan ke server pusat
- **Risiko Keamanan**: Data sensitif dapat bocor atau disalahgunakan
- **Keterbatasan Data**: Tidak semua rumah sakit bersedia berbagi data
- **Regulasi**: GDPR dan regulasi serupa membatasi penggunaan data kesehatan

### 1.3 Solusi yang Ditawarkan

Kami menawarkan solusi dengan dua teknologi utama:

**Federated Learning (FL)** adalah pendekatan machine learning di mana model dilatih secara terdistribusi di banyak perangkat. Setiap perangkat melatih model secara lokal dan hanya mengirimkan parameter model (bobot) ke server, bukan data asli.

**Blockchain** digunakan untuk:
- Menyimpan hash parameter model secara permanen
- Memverifikasi integritas update dari setiap klien
- Mendeteksi serangan poisoning attack
- Menyediakan audit trail yang transparan

<a name="metodologi"></a>
## 2. Metodologi

### 2.1 Arsitektur Sistem

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        SERVER (Aggregator)                              │
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────────────┐ │
│  │  FL Server      │  │  Blockchain     │  │  Model Aggregation      │ │
│  │  (FedAvg)       │  │  (Ledger)       │  │  (Parameter Update)     │ │
│  └────────┬────────┘  └────────┬────────┘  └────────────┬────────────┘ │
│           │                    │                        │              │
│           └────────────────────┼────────────────────────┘              │
│                                │                                       │
└────────────────────────────────┼───────────────────────────────────────┘
                                 │
                    (Hanya parameter model, BUKAN data)
                                 │
    ┌────────────────────────────┼────────────────────────────────────┐
    │                            │                                    │
    ▼                            ▼                                    ▼
┌──────────────┐      ┌──────────────┐                      ┌──────────────┐
│ Klien 1      │      │ Klien 2      │      ...             │ Klien N      │
│ (Perangkat   │      │ (Perangkat   │                      │ (Perangkat   │
│  Pasien 1)   │      │  Pasien 2)   │                      │  Pasien N)   │
│              │      │              │                      │              │
│ Data Lokal   │      │ Data Lokal   │                      │ Data Lokal   │
│ (TIDAK DI    │      │ (TIDAK DI    │                      │ (TIDAK DI    │
│  KIRIM!)     │      │  KIRIM!)     │                      │  KIRIM!)     │
└──────────────┘      └──────────────┘                      └──────────────┘
```

### 2.2 Algoritma Federated Averaging (FedAvg)

FedAvg adalah algoritma utama dalam Federated Learning:

1. **Inisialisasi**: Server mengirim model global ke semua klien
2. **Training Lokal**: Setiap klien melatih model pada data lokal
3. **Pengumpulan Parameter**: Klien mengirim parameter model ke server
4. **Agregasi**: Server menghitung weighted average dari semua parameter
5. **Update**: Server mengirim model global baru ke semua klien
6. **Pengulangan**: Proses diulang sampai konvergensi

**Rumus FedAvg:**

$$\theta_{global} = \sum_{k=1}^{K} \frac{n_k}{n_{total}} \cdot \theta_k$$

Di mana:
- $\theta_{global}$ adalah parameter model global
- $n_k$ adalah jumlah sampel di klien k
- $\theta_k$ adalah parameter model dari klien k

### 2.3 Keamanan Blockchain

Setiap parameter model yang dikirim ke server dicatat dalam blockchain dengan hash SHA-256. Proses ini memastikan:

- **Integritas**: Setiap perubahan pada parameter akan mengubah hash
- **Non-repudiation**: Klien tidak dapat menyangkal telah mengirim parameter tertentu
- **Audit Trail**: Semua transaksi dapat dilacak
- **Deteksi Serangan**: Parameter yang dimanipulasi akan terdeteksi

<a name="implementasi"></a>
## 3. Implementasi

### 3.1 Import Library

In [ ]:
# Import library yang diperlukan
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("Semua library berhasil diimport!")

### 3.2 Load dan Preprocessing Data

In [ ]:
# Path konfigurasi
BASE_DIR = r"d:\Kampus\Semester 6\Skripsi\Keperluan Skripsi"
DATA_PATH = os.path.join(BASE_DIR, 'Data-set', 'diabetes_prediction_dataset.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'FL_Blockchain_Diabetes_Type2', 'terminal', 'output')
IMG_DIR = os.path.join(BASE_DIR, 'FL_Blockchain_Diabetes_Type2', 'img')

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("\nKolom dalam dataset:")
print(df.columns.tolist())
print("\n5 baris pertama:")
df.head()

In [ ]:
# Cek distribusi target
print("Distribusi Diabetes:")
print(df['diabetes'].value_counts())
print(f"\nPersentase Diabetes: {df['diabetes'].mean()*100:.2f}%")

In [ ]:
# Preprocessing: Hapus duplikat
df_clean = df.drop_duplicates()
print(f"Setelah hapus duplikat: {len(df_clean):,} baris")

# Feature Engineering
# BMI Category
def categorize_bmi(bmi):
    if bmi < 18.5: return 0  # Underweight
    elif bmi < 25: return 1  # Normal
    elif bmi < 30: return 2  # Overweight
    else: return 3  # Obese

df_clean['bmi_category'] = df_clean['bmi'].apply(categorize_bmi)

# Age Category
def categorize_age(age):
    if age < 30: return 0  # Young
    elif age < 45: return 1  # Middle-aged
    elif age < 60: return 2  # Senior
    else: return 3  # Elderly

df_clean['age_category'] = df_clean['age'].apply(categorize_age)

# Risk Score
df_clean['risk_score'] = (
    df_clean['hypertension'] * 2 + 
    df_clean['heart_disease'] * 2 + 
    (df_clean['bmi'] > 30).astype(int) * 2 +
    (df_clean['HbA1c_level'] > 6.5).astype(int) * 3 +
    (df_clean['blood_glucose_level'] > 126).astype(int) * 3
)

# High Risk Indicator
df_clean['high_risk'] = ((df_clean['HbA1c_level'] >= 6.5) | (df_clean['blood_glucose_level'] >= 126)).astype(int)

print("Feature engineering selesai!")
print(f"Fitur baru: bmi_category, age_category, risk_score, high_risk")

In [ ]:
# Encoding variabel kategorikal
le_gender = LabelEncoder()
df_clean['gender'] = le_gender.fit_transform(df_clean['gender'])

le_smoking = LabelEncoder()
df_clean['smoking_history'] = le_smoking.fit_transform(df_clean['smoking_history'])

print("Encoding selesai!")
print(f"Gender: {dict(zip(le_gender.classes_, range(len(le_gender.classes_))))}")
print(f"Smoking: {dict(zip(le_smoking.classes_, range(len(le_smoking.classes_))))}")

In [ ]:
# Pilih fitur
features = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level',
            'blood_glucose_level', 'gender', 'smoking_history',
            'bmi_category', 'age_category', 'risk_score', 'high_risk']

X = df_clean[features].values
y = df_clean['diabetes'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]:,} sampel")
print(f"Testing: {X_test.shape[0]:,} sampel")

In [ ]:
# Oversampling untuk menangani data tidak seimbang
from sklearn.utils import resample

idx_0 = np.where(y_train == 0)[0]
idx_1 = np.where(y_train == 1)[0]

idx_1_oversample = resample(idx_1, replace=True, n_samples=len(idx_0), random_state=42)
idx_balanced = np.random.permutation(np.concatenate([idx_0, idx_1_oversample]))

X_train = X_train[idx_balanced]
y_train = y_train[idx_balanced]

print(f"Setelah oversampling: {len(y_train):,} sampel")
print(f"  - Non-Diabetes: {sum(y_train == 0):,}")
print(f"  - Diabetes: {sum(y_train == 1):,}")

In [ ]:
# Standarisasi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standarisasi selesai!")
print(f"Mean training: {X_train_scaled.mean(axis=0)[:3].round(4)}")
print(f"Std training: {X_train_scaled.std(axis=0)[:3].round(4)}")

### 3.3 Implementasi Federated Learning

In [ ]:
class FLClient:
    """
    Kelas FL Client - merepresentasikan perangkat pasien
    Data TIDAK PERNAH dikirim ke server, hanya parameter model
    """
    def __init__(self, cid, X, y):
        self.cid = cid
        self.X = X  # Data lokal (tersimpan di perangkat)
        self.y = y  # Label lokal
        self.n = len(y)
        
        # Model dilatih secara lokal
        self.model = LogisticRegression(max_iter=300, warm_start=True, random_state=42)
        
        # Inisialisasi dengan data dummy
        xi = np.vstack([X[:3], X[:3]])
        yi = np.array([0, 0, 0, 1, 1, 1])
        self.model.fit(xi, yi)
    
    def get_params(self):
        """Ambil parameter model (bukan data!)"""
        return {'coef': self.model.coef_.copy(), 'intercept': self.model.intercept_.copy()}
    
    def set_params(self, params):
        """Set parameter dari server"""
        self.model.coef_ = params['coef'].copy()
        self.model.intercept_ = params['intercept'].copy()
    
    def train(self, global_params):
        """Latih model pada data lokal"""
        self.set_params(global_params)
        self.model.fit(self.X, self.y)
        return {'params': self.get_params(), 'n': self.n}

print("Kelas FLClient didefinisikan!")

In [ ]:
class FLServer:
    """
    Kelas FL Server - mengaggregasi parameter dari klien
    """
    def __init__(self, n_features):
        self.params = {'coef': np.zeros((1, n_features)), 'intercept': np.zeros(1)}
    
    def fedavg(self, updates):
        """Federated Averaging - agregasi parameter"""
        total = sum(u['n'] for u in updates)
        coef = np.zeros_like(self.params['coef'])
        intercept = np.zeros_like(self.params['intercept'])
        
        for u in updates:
            weight = u['n'] / total
            coef += weight * u['params']['coef']
            intercept += weight * u['params']['intercept']
        
        self.params = {'coef': coef, 'intercept': intercept}
        return self.params
    
    def evaluate(self, X, y):
        """Evaluasi model global"""
        model = LogisticRegression(max_iter=1)
        model.classes_ = np.array([0, 1])
        model.coef_ = self.params['coef']
        model.intercept_ = self.params['intercept']
        
        y_pred = model.predict(X)
        y_prob = model.predict_proba(X)[:, 1]
        
        return {
            'Accuracy': accuracy_score(y, y_pred),
            'F1-Score': f1_score(y, y_pred),
            'AUC-ROC': roc_auc_score(y, y_prob)
        }

print("Kelas FLServer didefinisikan!")

In [ ]:
# Bagi data ke 5 klien (IID)
n_clients = 5
n_rounds = 20

idx = np.random.permutation(len(X_train_scaled))
parts = np.array_split(idx, n_clients)
client_data = [(X_train_scaled[p], y_train[p]) for p in parts]

# Buat klien
clients = [FLClient(i, *client_data[i]) for i in range(n_clients)]

# Buat server
server = FLServer(X_train_scaled.shape[1])

print(f"Dibuat {n_clients} klien FL")
print("Catatan: Data pasien TIDAK PERNAH meninggalkan perangkat!")

In [ ]:
# Jalankan Federated Learning
history = []

print("\n" + "="*60)
print("MENJALANKAN FEDERATED LEARNING")
print("="*60)

for r in range(1, n_rounds + 1):
    # Training lokal
    updates = [c.train(server.params) for c in clients]
    
    # Agregasi
    server.fedavg(updates)
    
    # Evaluasi
    metrics = server.evaluate(X_test_scaled, y_test)
    metrics['round'] = r
    history.append(metrics)
    
    if r % 5 == 0:
        print(f"Round {r:2d}: Acc={metrics['Accuracy']:.4f}, F1={metrics['F1-Score']:.4f}, AUC={metrics['AUC-ROC']:.4f}")

print("\nFederated Learning selesai!")

### 3.4 Implementasi Blockchain Security

In [ ]:
import hashlib
import json
import time

class Block:
    """Blok dalam blockchain"""
    def __init__(self, index, data, previous_hash):
        self.index = index
        self.timestamp = time.strftime('%Y-%m-%dT%H:%M:%S')
        self.data = data
        self.previous_hash = previous_hash
        self.hash = self._compute_hash()
    
    def _compute_hash(self):
        content = {
            'index': self.index,
            'timestamp': self.timestamp,
            'data': self.data,
            'previous_hash': self.previous_hash
        }
        content_str = json.dumps(content, sort_keys=True)
        return hashlib.sha256(content_str.encode()).hexdigest()

class Blockchain:
    """Ledger blockchain untuk FL"""
    def __init__(self):
        genesis = Block(0, {'info': 'Genesis Block - FL Diabetes'}, '0'*64)
        self.chain = [genesis]
        self.index_map = {}
    
    def add_block(self, client_id, round_num, model_hash, n_samples):
        data = {
            'client_id': client_id,
            'round': round_num,
            'model_hash': model_hash,
            'n_samples': n_samples,
            'verified': False
        }
        new_block = Block(len(self.chain), data, self.chain[-1].hash)
        self.chain.append(new_block)
        key = f"{client_id}_{round_num}"
        self.index_map[key] = {'model_hash': model_hash, 'block_index': new_block.index}
    
    def verify(self, client_id, round_num, received_hash):
        key = f"{client_id}_{round_num}"
        record = self.index_map.get(key)
        if not record: return False
        is_valid = record['model_hash'] == received_hash
        block_idx = record['block_index']
        self.chain[block_idx].data.update({'verified': is_valid, 'hash_received': received_hash})
        return is_valid

def hash_params(params):
    """Hash parameter model"""
    hasher = hashlib.sha256()
    hasher.update(params['coef'].astype(np.float32).tobytes())
    hasher.update(params['intercept'].astype(np.float32).tobytes())
    return hasher.hexdigest()

print("Blockchain classes didefinisikan!")

<a name="hasil"></a>
## 4. Hasil dan Pembahasan

### 4.1 Visualisasi Konvergensi FL

In [ ]:
# Plot konvergensi FL
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_df['round'], history_df['Accuracy'], 'b-o', linewidth=2)
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy per Round')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_df['round'], history_df['F1-Score'], 'g-o', linewidth=2)
axes[1].set_xlabel('Round')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1-Score per Round')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history_df['round'], history_df['AUC-ROC'], 'r-o', linewidth=2)
axes[2].set_xlabel('Round')
axes[2].set_ylabel('AUC-ROC')
axes[2].set_title('AUC-ROC per Round')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Federated Learning Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'fl_convergence_notebook.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Grafik konvergensi FL disimpan!")

### 4.2 Perbandingan dengan Centralized ML

In [ ]:
# Training model centralized untuk perbandingan
models = {
    'Logistic Regression': LogisticRegression(max_iter=300, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
}

centralized_results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    centralized_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob)
    })

centralized_df = pd.DataFrame(centralized_results)
print("Hasil Centralized ML:")
print(centralized_df.to_string(index=False))

In [ ]:
# Tambah hasil FL ke perbandingan
fl_result = {
    'Model': 'Federated Learning',
    'Accuracy': history_df.iloc[-1]['Accuracy'],
    'Precision': '-',
    'Recall': '-',
    'F1-Score': history_df.iloc[-1]['F1-Score'],
    'AUC-ROC': history_df.iloc[-1]['AUC-ROC']
}

comparison_df = pd.concat([centralized_df, pd.DataFrame([fl_result])], ignore_index=True)
print("\nPerbandingan Semua Model:")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].bar(comparison_df['Model'], comparison_df['Accuracy'], color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison', fontweight='bold')
axes[0].set_ylim(0.8, 1.0)
axes[0].tick_params(axis='x', rotation=45)

# F1-Score
axes[1].bar(comparison_df['Model'], comparison_df['F1-Score'], color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'])
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1-Score Comparison', fontweight='bold')
axes[1].set_ylim(0.4, 0.9)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Model Comparison: Centralized vs Federated Learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, 'model_comparison_notebook.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Grafik perbandingan disimpan!")

### 4.3 Demonstrasi Deteksi Serangan Blockchain

In [ ]:
# Demonstrasi deteksi serangan poisoning attack
print("="*60)
print("DEMONSTRASI DETEKSI POISONING ATTACK")
print("="*60)

# Setup ulang
clients_secure = [FLClient(i, *client_data[i]) for i in range(n_clients)]
ledger = Blockchain()
server_secure = FLServer(X_train_scaled.shape[1])

malicious_clients = [1]  # Klien 1 adalah attacker

for r in range(1, 6):
    print(f"\n--- Round {r} ---")
    
    updates = []
    for c in clients_secure:
        # Training lokal
        c.set_params(server_secure.params)
        c.model.fit(c.X, c.y)
        
        original_params = c.get_params()
        original_hash = hash_params(original_params)
        
        # Catat hash ke blockchain
        ledger.add_block(c.cid, r, original_hash, c.n)
        
        # Jika malicious, manipulasi parameter
        if c.cid in malicious_clients:
            print(f"  [ATTACK] Klien {c.cid} mengirim parameter MANIPULASI!")
            manipulated = {
                'coef': original_params['coef'] * -5,
                'intercept': original_params['intercept'] + 99
            }
            params_to_send = manipulated
        else:
            params_to_send = original_params
        
        sent_hash = hash_params(params_to_send)
        
        updates.append({
            'client_id': c.cid,
            'params': params_to_send,
            'hash': sent_hash,
            'n_samples': c.n
        })
    
    # Verifikasi dengan blockchain
    valid_updates = []
    for u in updates:
        is_valid = ledger.verify(u['client_id'], r, u['hash'])
        if is_valid:
            print(f"  [Server] Klien {u['client_id']}: VALID")
            valid_updates.append(u)
        else:
            print(f"  [Server] Klien {u['client_id']}: REJECTED - Hash tidak cocok!")
    
    # Agregasi hanya dari update valid
    if valid_updates:
        total = sum(u['n_samples'] for u in valid_updates)
        coef = np.zeros_like(server_secure.params['coef'])
        intercept = np.zeros_like(server_secure.params['intercept'])
        for u in valid_updates:
            weight = u['n_samples'] / total
            coef += weight * u['params']['coef']
            intercept += weight * u['params']['intercept']
        server_secure.params = {'coef': coef, 'intercept': intercept}
    
    # Evaluasi
    metrics = server_secure.evaluate(X_test_scaled, y_test)
    print(f"  [Server] Accuracy: {metrics['Accuracy']:.4f}")

<a name="kesimpulan"></a>
## 5. Kesimpulan

### 5.1 Keunggulan Sistem

Sistem prediksi diabetes dengan Federated Learning dan Blockchain yang kami kembangkan memiliki keunggulan:

1. **Privasi Terjamin**: Data pasien tidak pernah dikirim ke server pusat. Hanya parameter model yang diaggregasi.

2. **Keamanan Blockchain**: Setiap update parameter dicatat dalam blockchain dengan hash SHA-256, memastikan integritas data.

3. **Deteksi Serangan**: Sistem dapat mendeteksi poisoning attack dengan membandingkan hash yang dikirim dengan hash yang tercatat di blockchain.

4. **Desentralisasi**: Tidak ada server sentral yang menyimpan data sensitif. Training terjadi di perangkat masing-masing pasien.

5. **Transparansi**: Audit trail yang lengkap memungkinkan verifikasi setiap update model.

### 5.2 Implikasi untuk Dunia Medis

Sistem ini dapat diterapkan untuk:

- **Rumah Sakit**: Berbagi model tanpa berbagi data pasien
- **Penelitian**: Kolaborasi multi-pusat dengan privasi terjaga
- **Aplikasi Kesehatan**: Prediksi diabetes langsung di perangkat pengguna
- **Telemedicine**: Konsultasi AI dengan keamanan data pasien

### 5.3 Keterbatasan dan Pengembangan Lanjutan

Keterbatasan:
- Kompleksitas komputasi di perangkat klien
- Latency komunikasi antar klien dan server
- Perlu infrastruktur yang mendukung

Pengembangan lanjutan:
- Implementasi differential privacy
- Integrasi dengan sistem EHR yang ada
- Optimasi untuk perangkat mobile
- Pengembangan smart contract untuk otomatisasi

---

## Referensi

1. McMahan, B., et al. (2017). Communication-Efficient Learning of Deep Networks from Decentralized Data.
2. Kairouz, P., et al. (2021). Advances and Open Problems in Federated Learning.
3. Nakamoto, S. (2008). Bitcoin: A Peer-to-Peer Electronic Cash System.
4. International Diabetes Federation. (2021). IDF Diabetes Atlas 10th Edition.

In [ ]:
print("="*60)
print("NOTEBOOK SELESAI!")
print("="*60)
print("\nSistem prediksi diabetes dengan Federated Learning dan Blockchain")
print("telah berhasil diimplementasikan dan didemonstrasikan.")
print("\nKeunggulan sistem:")
print("1. Data pasien TIDAK PERNAH dikirim ke server pusat")
print("2. Training terjadi di dalam device pasien")
print("3. Blockchain memastikan integritas model update")
print("4. Deteksi otomatis serangan poisoning attack")
print("5. Transparansi dan audit trail untuk setiap update")